In [8]:
import sys
!{sys.executable} -m pip install opencv-python numpy mediapipe
import cv2
import numpy as np
import mediapipe as mp


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
def show_hsv_range(lower, upper, title="HSV Range"):
    h = np.linspace(0, 179, 360)
    s = np.linspace(0, 255, 256)
    H, S = np.meshgrid(h, s)
    V = np.ones_like(H) * 255

    hsv = np.stack([H, S, V], axis=-1).astype(np.uint8)
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    mask = cv2.inRange(hsv, lower, upper) 
    result = cv2.bitwise_and(bgr, bgr, mask=mask)

    cv2.imshow(f"{title} - Selected Colors", result)

    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [10]:
lower_red1 = np.array([0, 140, 120])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([165, 120, 120])
upper_red2 = np.array([180, 255, 255])

lower_yellow = np.array([20, 150, 150])
upper_yellow = np.array([35, 255, 255])

lower_green  = np.array([40, 80,  80])
upper_green  = np.array([85, 255, 255])

lower_blue = np.array([100, 80, 80])
upper_blue = np.array([130, 255, 255])



In [13]:
def is_circle(contour, min_area=1000, circularity_threshold=0.85):
    area = cv2.contourArea(contour)
    if area < min_area:
        return False
    perimeter = cv2.arcLength(contour, True)
    if perimeter == 0:
        return False
    
    circularity = 4 * np.pi * area / (perimeter * perimeter)
    return circularity > circularity_threshold

In [15]:
def process_camera(camera_id=0):
    """
    Захват видео с веб-камеры в реальном времени.
    Нажмите 'q' для выхода.
    """

    cap = cv2.VideoCapture(camera_id)
    
    if not cap.isOpened():
        print("Ошибка: не удалось открыть камеру!")
        return
    
    print("Обработка камеры началась... Нажмите 'q' для выхода")
    
    kernel = np.ones((5, 5), np.uint8)
    
    status_history = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Ошибка захвата кадра с камеры")
            break
        
        start_time = cv2.getTickCount()
        
        img = frame.copy()
        img = cv2.GaussianBlur(img, (7, 7), 0)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        
        mask_red = cv2.bitwise_or(
            cv2.inRange(hsv, lower_red1, upper_red1),
            cv2.inRange(hsv, lower_red2, upper_red2)
        )
        mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)
        mask_green = cv2.inRange(hsv, lower_green, upper_green)
        mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

        mask_red_clean = cv2.morphologyEx(mask_red, cv2.MORPH_CLOSE, kernel, iterations=2)
        mask_yellow_clean = cv2.morphologyEx(mask_yellow, cv2.MORPH_CLOSE, kernel, iterations=2)
        mask_green_clean = cv2.morphologyEx(mask_green, cv2.MORPH_CLOSE, kernel, iterations=2)
        mask_blue_clean = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel, iterations=2)
        
        # 6. Находим контуры
        contours_red = cv2.findContours(mask_red_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        contours_yellow = cv2.findContours(mask_yellow_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        contours_green = cv2.findContours(mask_green_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        contours_blue = cv2.findContours(mask_blue_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]

        def find_circles(contours, color_bgr):
            count = 0
            for cnt in contours:
                cv2.drawContours(img, [cnt], -1, (255, 0, 0), 1)
                area = cv2.contourArea(cnt)
                perimeter = cv2.arcLength(cnt, True)
                
                if perimeter == 0:
                    continue
                
                circularity = 4 * np.pi * area / (perimeter * perimeter)
                
                if is_circle(cnt):
                    count += 1
                    
                    (x, y), radius = cv2.minEnclosingCircle(cnt)
                    center = (int(x), int(y))
                    cv2.circle(img, center, int(radius), color_bgr, 4)
                    cv2.circle(img, center, 6, (255, 255, 255), -1)
                    text = f"A:{int(area)} P:{int(perimeter)} C:{circularity:.2f}"
                    text_x = max(0, center[0] - 80)
                    text_y = max(30, center[1] - int(radius) - 10)
                    
                    cv2.putText(img, text,
                                (text_x, text_y),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.5,
                                color_bgr,
                                2,
                                cv2.LINE_AA)
            
            return count
        
        red_count = find_circles(contours_red, (0, 0, 255))
        yellow_count = find_circles(contours_yellow, (0, 255, 255))
        green_count = find_circles(contours_green, (0, 255, 0))
        blue_count = find_circles(contours_blue, (255, 0, 0))  
        
        if green_count > 0:
            status = "GO"
            text_color = (0, 255, 0)
        elif red_count > 0 and yellow_count > 0:
            status = "GET READY"
            text_color = (0, 255, 255)
        elif red_count > 0:
            status = "STOP"
            text_color = (0, 0, 255)
        elif blue_count > 0:
            status = "WAIT"  # Или "BLUE"
            text_color = (255, 0, 0)
        else:
            status = ""
            text_color = (255, 255, 255)
        
        status_history.append(status)
        if len(status_history) > 10:  
            status_history.pop(0)
        
        if status_history:
            most_common_status = max(set(status_history), key=status_history.count)
            if most_common_status:
                cv2.putText(img, most_common_status, (50, 100), 
                           cv2.FONT_HERSHEY_SIMPLEX, 2.5, text_color, 6, cv2.LINE_AA)
        
        
        cv2.imshow("Traffic Light Detector (Camera)", img)
        
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    
    cap.release()
    cv2.destroyAllWindows()
    print("Обработка камеры завершена.")


if __name__ == "__main__":
    process_camera(0)

Обработка камеры началась... Нажмите 'q' для выхода
Обработка камеры завершена.


In [7]:
def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print("Ошибка: не удалось открыть видео файл")
        return
    
    print("Обработка видео началась... Нажмите 'q' для выхода")
    
    speed_factor = 0.8
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    delay = int(1000 / (original_fps * speed_factor))
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Видео закончилось")
            break
        
        start_time = cv2.getTickCount()
                
        img = frame.copy()
        img = cv2.GaussianBlur(img, (7, 7), 0)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        kernel = np.ones((5, 5), np.uint8)

        mask_red   = cv2.bitwise_or(
            cv2.inRange(hsv, lower_red1, upper_red1),
            cv2.inRange(hsv, lower_red2, upper_red2)
        )
        mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)
        mask_green  = cv2.inRange(hsv, lower_green,  upper_green)
        
        mask_red_clean    = cv2.morphologyEx(mask_red,    cv2.MORPH_CLOSE, kernel, iterations=2)
        mask_yellow_clean = cv2.morphologyEx(mask_yellow, cv2.MORPH_CLOSE, kernel, iterations=2)
        mask_green_clean  = cv2.morphologyEx(mask_green,  cv2.MORPH_CLOSE, kernel, iterations=2)

        contours_red    = cv2.findContours(mask_red_clean,    cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        contours_yellow = cv2.findContours(mask_yellow_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        contours_green  = cv2.findContours(mask_green_clean,  cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        
        def find_circles(contours, color_bgr):
            count = 0
            for cnt in contours:
                cv2.drawContours(img, [cnt], -1, (255, 0, 0), 1)
                area = cv2.contourArea(cnt)
                perimeter = cv2.arcLength(cnt, True)

                if perimeter == 0:
                    continue

                circularity = 4 * np.pi * area / (perimeter * perimeter)

                if is_circle(cnt):
                    count += 1

                    (x, y), radius = cv2.minEnclosingCircle(cnt)
                    center = (int(x), int(y))
                    cv2.circle(img, center, int(radius), color_bgr, 4)
                    cv2.circle(img, center, 6, (255, 255, 255), -1)
                    text = f"A:{int(area)} P:{int(perimeter)} C:{circularity:.2f}"
                    text_x = max(0, center[0] - 80)
                    text_y = max(30, center[1] - int(radius) - 10)

                    cv2.putText(img, text,
                                (text_x, text_y),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.5,
                                color_bgr,
                                2,
                                cv2.LINE_AA)

            return count
                
        red_count    = find_circles(contours_red,    (0, 0, 255))
        yellow_count = find_circles(contours_yellow, (0, 255, 255))
        green_count  = find_circles(contours_green,  (0, 255, 0))
        
        if green_count > 0:
            status = "GO"
            text_color = (0, 255, 0)
        elif red_count > 0 and yellow_count > 0:
            status = "GET READY"
            text_color = (0, 255, 255)
        elif red_count > 0:
            status = "STOP"
            text_color = (0, 0, 255)
        else:
            status = ""
            text_color = (255, 255, 255)
        
        if status:
            cv2.putText(img, status, (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2.5, text_color, 6, cv2.LINE_AA)
        
        cv2.imshow("Traffic Light Detector", img)
        
        processing_time = (cv2.getTickCount() - start_time) / cv2.getTickFrequency() * 1000
        wait_time = max(1, int(delay - processing_time))
        
        if cv2.waitKey(wait_time) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print("Обработка завершена")

In [8]:
traffic_video_file = "traffic.mp4"

process_video(traffic_video_file)

Обработка видео началась... Нажмите 'q' для выхода
Видео закончилось
Обработка завершена


In [9]:
BaseOptions = mp.tasks.BaseOptions
ObjectDetector = mp.tasks.vision.ObjectDetector
ObjectDetectorOptions = mp.tasks.vision.ObjectDetectorOptions
VisionRunningMode = mp.tasks.vision.RunningMode

In [10]:
options = ObjectDetectorOptions(
    base_options=BaseOptions(
        model_asset_path='efficientdet_lite2.tflite'
    ),
    running_mode=VisionRunningMode.VIDEO,
    score_threshold=0.40,
    max_results=10
)

In [11]:
def count_people_in_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Ошибка открытия видео!")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 30

    frame_timestamp_ms = 0
    people_count = 0

    print("Обработка видео началась... Нажмите 'q' для выхода")

    with ObjectDetector.create_from_options(options) as detector:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Видео закончилось.")
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            detection_result = detector.detect_for_video(mp_image, frame_timestamp_ms)
            frame_timestamp_ms += int(1000 / fps)

            current_people = 0
            for detection in detection_result.detections:
                category = detection.categories[0]
                if category.category_name == "person" and category.score >= 0.35:
                    current_people += 1

                    bbox = detection.bounding_box
                    start_point = (int(bbox.origin_x), int(bbox.origin_y))
                    end_point = (int(bbox.origin_x + bbox.width), int(bbox.origin_y + bbox.height))
                    cv2.rectangle(frame, start_point, end_point, (0, 255, 0), 3)

                    label = f"Person: {category.score:.2f}"
                    cv2.putText(frame, label, (start_point[0], start_point[1] - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            people_count = current_people

            cv2.putText(frame, f"Total People: {people_count}", (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 0, 255), 4, cv2.LINE_AA)
            cv2.imshow("Intelligent People Counter", frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
    print("Обработка завершена.")

In [12]:
auditorium_video_file = "auditorium.mp4"

count_people_in_video(auditorium_video_file)

Обработка видео началась... Нажмите 'q' для выхода
Обработка завершена.
